In [ ]:
JOBS_GPU0 = ['diagnostics']
JOBS_GPU1 = []
JOBS_CPU = []
RUN_TAG = 'vna_v4_exact_diagnostics'
PRESET = 'EXACT_DIAGNOSTICS'
REPO_COMMIT = 'b1828ac7cf9fe50a3d755d9a88f92e8a36e6be55'
SOURCE_DIGEST = None
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 5_000_000_000

MODE = 'profile'  # then 'nan'; each uses one T4


# T29–T31: one-T4 diagnostics
Attach vna4_main.tar.gz and CIFAR-100. Code loads directly from the pinned Git commit; no source ZIP is needed. Run profile first; then A–D. M1 stays held for report review.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/vna4_main")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if branch == REPO_COMMIT:
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", REPO_COMMIT], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", REPO_COMMIT], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
DATA_ROOT = Path('/kaggle/input')
MNIST_ROOT = next((p.parent for p in Path('/kaggle/input').rglob('mnist_') if p.is_file()),None)
if JOBS_GPU0 or JOBS_GPU1:
    import torch
    if torch.cuda.device_count() < 1:
        raise RuntimeError(f"Select at least one T4; found {torch.cuda.device_count()} GPU(s)")
    required_cifar_files = {"train", "test", "meta"}
    cifar_dirs = []
    for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
        path = Path(directory)
        if (path.name == "cifar-100-python" and
                required_cifar_files.issubset(filenames)):
            cifar_dirs.append(path.parent)
    cifar_dirs = sorted(set(cifar_dirs), key=str)
    if not cifar_dirs:
        visible = []
        for directory, dirnames, filenames in os.walk(
                "/kaggle/input", followlinks=True):
            visible.append({
                "directory": directory,
                "directories": sorted(dirnames)[:10],
                "files": sorted(filenames)[:10],
            })
            if len(visible) == 20:
                break
        raise FileNotFoundError(
            "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
            f"Visible input inventory: {visible}")
    DATA_ROOT = cifar_dirs[0]
    print("CIFAR-100 root:", DATA_ROOT)
    
if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]



subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quadprog"], check=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), STAGE_B, 'outputs')
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
from experiments.io_utils import atomic_json_save
archives = sorted(Path('/kaggle/input').rglob('vna4_main.tar.gz'))
if len(archives) != 1: raise FileNotFoundError('Attach exactly one old vna4_main.tar.gz containing full task-1 latest.pt')
OLD_REPO = REPO.with_name('counterfactual-projection-d88abc4')
if not OLD_REPO.exists():
    subprocess.run(['git','clone','--shared',str(REPO),str(OLD_REPO)],check=True)
    subprocess.run(['git','-C',str(OLD_REPO),'checkout','--detach','d88abc4'],check=True)
    for upstream in (REPO/'third_party').iterdir():
        destination = OLD_REPO/'third_party'/upstream.name
        if upstream.is_dir() and not destination.exists(): destination.symlink_to(upstream, target_is_directory=True)
script = REPO/'diagnostics/exact_solver.py'
branches = ('B','D') if MODE == 'profile' else ('A','B','C','D') if MODE == 'nan' else ()
if not branches: raise ValueError('MODE must be profile or nan')
try:
    for branch in branches:
        tree = OLD_REPO if branch in ('B','C') else REPO
        directory = STAGE_B/'exact_diagnostics'/MODE/branch
        directory.mkdir(parents=True,exist_ok=True)
        command = [sys.executable,'-u',str(script),'--repo',str(tree),'--branch',branch,
            '--archive',str(archives[0]),'--data-dir',str(DATA_ROOT),'--out',str(directory)]
        if MODE == 'profile': command.append('--profile')
        if not launch(command,0,f'{MODE}_{branch}',RUN_ROOT/f'{MODE}_{branch}.log'): break
        archive_outputs()
finally:
    archive_outputs()
print('M1 remains held: review the actual profiles and A–D diagnostic reports.')


In [ ]:
for path in (STAGE_B/'exact_diagnostics').rglob('diagnostic.json'): print(path, path.read_text())
archive_outputs()
